In [1]:

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')


def fetch_earthquake_data(start_date, end_date, min_magnitude=2.5):
    base_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

    params = {
        "format": "geojson",
        "starttime": start_date,
        "endtime": end_date,
        "minmagnitude": min_magnitude,
        "orderby": "time",
        "limit": 20000,
    }

    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        print(f"   Retrieved {len(data.get('features', []))} events")
        return data
    else:
        print(f"   Error: {response.status_code} - {response.text[:200]}")
        return None

all_features = []

end_date = datetime.now()

date_ranges = []
for i in range(4):
    chunk_end = end_date - timedelta(days=180 * i)
    chunk_start = end_date - timedelta(days=180 * (i + 1))
    date_ranges.append((chunk_start.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))


for start, end in date_ranges:
    data = fetch_earthquake_data(start, end, min_magnitude=2.5)
    if data and "features" in data:
        all_features.extend(data["features"])


raw_data = {"features": all_features}
def parse_earthquake_data(features):
    """Convert raw GeoJSON features into a clean DataFrame."""
    parsed_data = []

    for feature in features:
        props = feature["properties"]
        coords = feature["geometry"]["coordinates"]

        parsed_data.append(
            {
                # Core
                "magnitude": props.get("mag"),
                "place": props.get("place"),
                "time": props.get("time"),
                "updated": props.get("updated"),
                "tsunami": props.get("tsunami"),
                "significance": props.get("sig"),
                "alert": props.get("alert"),
                # Geographic
                "longitude": coords[0] if len(coords) > 0 else None,
                "latitude": coords[1] if len(coords) > 1 else None,
                "depth": coords[2] if len(coords) > 2 else None,
                # Quality
                "nst": props.get("nst"),
                "dmin": props.get("dmin"),
                "rms": props.get("rms"),
                "gap": props.get("gap"),
                # Intensity
                "felt": props.get("felt"),
                "cdi": props.get("cdi"),
                "mmi": props.get("mmi"),
                # Metadata
                "mag_type": props.get("magType"),
                "event_type": props.get("type"),
                "status": props.get("status"),
                "net": props.get("net"),
            }
        )

    return pd.DataFrame(parsed_data)


df = parse_earthquake_data(raw_data["features"])

# Time features
df["datetime"] = pd.to_datetime(df["time"], unit="ms")
df["date"] = df["datetime"].dt.date
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek


# Risk level label
def classify_risk(row):
    mag = row["magnitude"]
    if pd.isna(mag):
        return "Unknown"
    elif mag >= 6.0:
        return "High"
    elif mag >= 4.5:
        return "Medium"
    else:
        return "Low"


df["risk_level"] = df.apply(classify_risk, axis=1)
risk_mapping = {"Low": 0, "Medium": 1, "High": 2, "Unknown": -1}
df["risk_numeric"] = df["risk_level"].map(risk_mapping)

# Distance from equator
df["dist_from_equator"] = df["latitude"].abs()

# Depth category
df["depth_category"] = pd.cut(
    df["depth"], bins=[0, 70, 300, 700], labels=["Shallow", "Intermediate", "Deep"]
)



# Statistical Analysis on the data
from scipy.stats import ttest_ind, f_oneway, pearsonr, chi2_contingency


def cohens_d(x, y):
    """Cohen's d for two independent samples."""
    x = np.asarray(x)
    y = np.asarray(y)
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_var = ((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / dof
    pooled_std = np.sqrt(pooled_var)
    return (x.mean() - y.mean()) / pooled_std


def cramers_v_from_chi2(chi2, contingency_table):
    """Cramer's V for association in contingency table."""
    n = contingency_table.values.sum()
    r, c = contingency_table.shape
    return np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

key_vars = ["magnitude", "depth", "significance", "latitude", "longitude"]
print(df[key_vars].describe().round(3))

print("\nStats by Risk Level (magnitude, depth, significance):")
print(
    df.groupby("risk_level")[
        ["magnitude", "depth", "significance"]
    ].agg(["mean", "std", "count"]).round(3)
)

# T-test: depth (Low vs Medium risk)
low_depth = df[df["risk_level"] == "Low"]["depth"].dropna()
med_depth = df[df["risk_level"] == "Medium"]["depth"].dropna()

t_stat, p_val = ttest_ind(low_depth, med_depth)
print(f"Low Risk:    n={len(low_depth)}, mean={low_depth.mean():.2f} km")
print(f"Medium Risk: n={len(med_depth)}, mean={med_depth.mean():.2f} km")
print(f"t-statistic={t_stat:.4f}, p-value={p_val:.2e}")

# Effect size
d_depth = cohens_d(low_depth, med_depth)
print(f"Cohen's d (Low vs Medium depth): {d_depth:.3f}")

# ANOVA: magnitude across depth categories
print("ANOVA: Magnitude Across Depth Categories")

shallow_mag = df[df["depth_category"] == "Shallow"]["magnitude"].dropna()
inter_mag = df[df["depth_category"] == "Intermediate"]["magnitude"].dropna()
deep_mag = df[df["depth_category"] == "Deep"]["magnitude"].dropna()

f_stat, p_val_anova = f_oneway(shallow_mag, inter_mag, deep_mag)
print(f"F-statistic={f_stat:.4f}, p-value={p_val_anova:.2e}")

# Eta-squared effect size
all_mag = np.concatenate([shallow_mag.values, inter_mag.values, deep_mag.values])
grand_mean = all_mag.mean()
groups = [shallow_mag.values, inter_mag.values, deep_mag.values]
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((all_mag - grand_mean) ** 2).sum()
eta_squared = ss_between / ss_total


# Correlation: magnitude vs significance
mag = df["magnitude"].dropna()
sig = df["significance"].dropna()

min_len = min(len(mag), len(sig))
mag = mag.iloc[:min_len]
sig = sig.iloc[:min_len]
corr_coef, p_val_corr = pearsonr(mag, sig)
print(f"Pearson r={corr_coef:.4f}, p-value={p_val_corr:.2e}")

# Chi-square: depth category vs risk level
contingency = pd.crosstab(df["depth_category"], df["risk_level"])
chi2, p_val_chi2, dof, expected = chi2_contingency(contingency)
print(contingency)
print(f"Chi²={chi2:.4f}, dof={dof}, p-value={p_val_chi2:.2e}")

cramers_v = cramers_v_from_chi2(chi2, contingency)
print(f"Cramer's V for depth_category vs risk_level: {cramers_v:.3f}")

# Machine Learning & Model-Level Tests

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.utils import class_weight
from scipy.stats import ttest_rel
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Data preparation for ML

feature_columns = [
    "depth",
    "latitude",
    "longitude",
    "nst",
    "dmin",
    "rms",
    "gap",
    "hour",
    "day_of_week",
    "month",
    "dist_from_equator",
]

ml_df = df[feature_columns + ["magnitude", "risk_level", "risk_numeric"]].dropna()


X = ml_df[feature_columns].values
y_classification = ml_df["risk_numeric"].values
y_regression = ml_df["magnitude"].values

valid_mask = y_classification >= 0
X_clf = X[valid_mask]
y_clf = y_classification[valid_mask]

# PCA
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X)

pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_scaled)

# Train/test split & scaling

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)


# Classification models
classification_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7),
}

classification_results = {}

for name, model in classification_models.items():
    print(f"\n🔹 Training {name}...")
    model.fit(X_train_clf_scaled, y_train_clf)
    y_pred = model.predict(X_test_clf_scaled)

    acc = accuracy_score(y_test_clf, y_pred)
    f1 = f1_score(y_test_clf, y_pred, average="weighted")

    classification_results[name] = {
        "model": model,
        "Accuracy_test": acc,
        "F1_test": f1,
    }

# Deep learning classifier (DNN)
num_classes = len(np.unique(y_train_clf))

# Class weights for imbalanced classes
unique_classes = np.unique(y_train_clf)
class_weights_array = class_weight.compute_class_weight(
    class_weight="balanced", classes=unique_classes, y=y_train_clf
)
class_weights = {int(c): float(w) for c, w in zip(unique_classes, class_weights_array)}

clf_dnn = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train_clf_scaled.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation="softmax"),
    ]
)

clf_dnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop_clf = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history_clf_dnn = clf_dnn.fit(
    X_train_clf_scaled,
    y_train_clf,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop_clf],
    class_weight=class_weights,
    verbose=1,
)

# Evaluate DNN classifier
y_pred_dnn_proba = clf_dnn.predict(X_test_clf_scaled)
y_pred_dnn = np.argmax(y_pred_dnn_proba, axis=1)

dnn_acc = accuracy_score(y_test_clf, y_pred_dnn)
dnn_f1 = f1_score(y_test_clf, y_pred_dnn, average="weighted")
print(f"   Accuracy: {dnn_acc:.4f}")
print(f"   F1-score: {dnn_f1:.4f}")

classification_results["DNN Classifier"] = {
    "model": clf_dnn,
    "Accuracy_test": dnn_acc,
    "F1_test": dnn_f1,
}

# Per-class metrics & confusion matrices

risk_names = ["Low", "Medium", "High"]

for name, result in classification_results.items():
    model = result["model"]

    if name == "DNN Classifier":
        y_pred_proba = model.predict(X_test_clf_scaled)
        y_pred = np.argmax(y_pred_proba, axis=1)
    else:
        y_pred = model.predict(X_test_clf_scaled)

    f1_macro = f1_score(y_test_clf, y_pred, average="macro")
    classification_results[name]["F1_macro_test"] = f1_macro
    print(f"Macro F1 (test): {f1_macro:.4f}")

    cm = confusion_matrix(y_test_clf, y_pred, labels=[0, 1, 2])
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=risk_names, columns=risk_names))

# Regression models
regression_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=200, random_state=42
    ),
}

regression_results = {}

for name, model in regression_models.items():
    print(f"\n Training {name} ")
    model.fit(X_train_reg_scaled, y_train_reg)
    y_pred = model.predict(X_test_reg_scaled)

    mse = mean_squared_error(y_test_reg, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)

    regression_results[name] = {
        "model": model,
        "MSE_test": mse,
        "RMSE_test": rmse,
        "MAE_test": mae,
        "R2_test": r2,
    }

    print(f"   RMSE (test): {rmse:.4f}")
    print(f"   MAE  (test): {mae:.4f}")
    print(f"   R²   (test): {r2:.4f}")

# Deep learning regression (DNN)

model_dnn = Sequential(
    [
        Dense(128, activation="relu", input_shape=(X_train_reg_scaled.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ]
)

model_dnn.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

early_stop_dnn = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history_dnn = model_dnn.fit(
    X_train_reg_scaled,
    y_train_reg,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop_dnn],
    verbose=1,
)

y_pred_dnn_reg = model_dnn.predict(X_test_reg_scaled).flatten()

dnn_mse = mean_squared_error(y_test_reg, y_pred_dnn_reg)
dnn_rmse = np.sqrt(dnn_mse)
dnn_mae = mean_absolute_error(y_test_reg, y_pred_dnn_reg)
dnn_r2 = r2_score(y_test_reg, y_pred_dnn_reg)
print(f"   RMSE: {dnn_rmse:.4f}")
print(f"   MAE : {dnn_mae:.4f}")
print(f"   R²  : {dnn_r2:.4f}")

regression_results["DNN Regressor"] = {
    "model": model_dnn,
    "MSE_test": dnn_mse,
    "RMSE_test": dnn_rmse,
    "MAE_test": dnn_mae,
    "R2_test": dnn_r2,
}
def evaluate_classification_cv(base_model, X, y, cv):
    acc_scores, f1_scores = [], []
    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        model = clone(base_model)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        acc_scores.append(accuracy_score(y_te, y_pred))
        f1_scores.append(f1_score(y_te, y_pred, average="weighted"))
    return np.array(acc_scores), np.array(f1_scores)

def evaluate_regression_cv(base_model, X, y, cv):
    rmse_scores, r2_scores = [], []
    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        model = clone(base_model)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        rmse_scores.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        r2_scores.append(r2_score(y_te, y_pred))
    return np.array(rmse_scores), np.array(r2_scores)


# Test 1: Classification - Random Forest vs Logistic Regression
print("\n Classification: Random Forest vs Logistic Regression (5-fold CV)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_clf_model = classification_results["Random Forest"]["model"]
log_clf_model = classification_results["Logistic Regression"]["model"]

rf_acc, rf_f1 = evaluate_classification_cv(rf_clf_model, X_clf, y_clf, skf)
log_acc, log_f1 = evaluate_classification_cv(log_clf_model, X_clf, y_clf, skf)

print("\nMean ± SD Accuracy:")
print(f"   Random Forest       : {rf_acc.mean():.4f} ± {rf_acc.std():.4f}")
print(f"   Logistic Regression : {log_acc.mean():.4f} ± {log_acc.std():.4f}")

stat_acc, p_acc = ttest_rel(rf_acc, log_acc)
print("\nPaired t-test on Accuracy (RF vs LR):")
print(f"   t-statistic: {stat_acc:.4f}")
print(f"   p-value   : {p_acc:.4e}")

# Test 2: Regression - Random Forest Regressor vs Gradient Boosting Regressor
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rf_reg_model = regression_results["Random Forest Regressor"]["model"]
gb_reg_model = regression_results["Gradient Boosting Regressor"]["model"]

rf_rmse, rf_r2 = evaluate_regression_cv(rf_reg_model, X, y_regression, kf)
gb_rmse, gb_r2 = evaluate_regression_cv(gb_reg_model, X, y_regression, kf)

print("\nMean ± SD RMSE:")
print(f"   Random Forest Regressor : {rf_rmse.mean():.4f} ± {rf_rmse.std():.4f}")
print(f"   Gradient Boosting       : {gb_rmse.mean():.4f} ± {gb_rmse.std():.4f}")

stat_rmse, p_rmse = ttest_rel(rf_rmse, gb_rmse)
print("\nPaired t-test on RMSE (RF vs GB):")
print(f"   t-statistic: {stat_rmse:.4f}")
print(f"   p-value   : {p_rmse:.4e}")

#LSTM
from sklearn.preprocessing import MinMaxScaler
daily_counts = df.groupby("date").size().reset_index(name="count")
daily_counts["date"] = pd.to_datetime(daily_counts["date"])
daily_counts = daily_counts.sort_values("date").reset_index(drop=True)
# Scale counts to [0, 1] for LSTM
values = daily_counts["count"].values.reshape(-1, 1)
scaler_ts = MinMaxScaler()
counts_scaled = scaler_ts.fit_transform(values)

SEQ_LEN = 7 

def create_sequences(data, seq_length):
    Xs, ys = [], []
    for i in range(len(data) - seq_length):
        Xs.append(data[i : i + seq_length])
        ys.append(data[i + seq_length])
    return np.array(Xs), np.array(ys)


X_ts, y_ts = create_sequences(counts_scaled, SEQ_LEN)
X_ts = X_ts.reshape((X_ts.shape[0], X_ts.shape[1], 1))

# Time-respecting split: first 80% train, last 20% test
split_idx = int(len(X_ts) * 0.8)
X_train_ts, X_test_ts = X_ts[:split_idx], X_ts[split_idx:]
y_train_ts, y_test_ts = y_ts[:split_idx], y_ts[split_idx:]
model_lstm = Sequential(
    [
        LSTM(32, return_sequences=True, input_shape=(SEQ_LEN, 1)),
        Dropout(0.2),
        LSTM(16),
        Dense(1),
    ]
)

model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

early_stop_lstm = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train_ts,
    y_train_ts,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1,
)

y_pred_scaled = model_lstm.predict(X_test_ts)

# Invert scaling
y_pred_counts = scaler_ts.inverse_transform(y_pred_scaled).flatten()
y_test_counts = scaler_ts.inverse_transform(y_test_ts).flatten()
y_train_counts = scaler_ts.inverse_transform(y_train_ts).flatten()

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lstm_mse = mean_squared_error(y_test_counts, y_pred_counts)
lstm_rmse = np.sqrt(lstm_mse)
lstm_mae = mean_absolute_error(y_test_counts, y_pred_counts)
lstm_r2 = r2_score(y_test_counts, y_pred_counts)
print(f"   RMSE: {lstm_rmse:.4f}")
print(f"   MAE : {lstm_mae:.4f}")
print(f"   R²  : {lstm_r2:.4f}")

# #Another setup for LSTM
# REGIONAL LSTM TIME SERIES (UNIVARIATE DAILY COUNTS)


# from sklearn.preprocessing import MinMaxScaler

# # Create a simple "region" column if it doesn't exist already
# if "region" not in df.columns:
#     # Try to extract text after "of " (e.g. "100km NW of Tokyo, Japan")
#     df["region"] = df["place"].str.extract(r"of (.+)$")[0]
#     # Fallback: if nothing extracted, use the full place string
#     df["region"] = df["region"].fillna(df["place"])

# REGION_NAME = "Japan"   # e.g. "Japan", "Alaska", "California", etc.
# SEQ_LEN = 7            # days of history → predict next day
# region_mask = df["region"].str.contains(REGION_NAME, case=False, na=False)
# region_df = df[region_mask].copy()

# if region_df.empty:
#     raise ValueError(
#         f"No earthquakes found matching region filter '{REGION_NAME}'. "
#         "Try a broader keyword like 'Alaska', 'Russia', 'Fiji', or inspect df['region'].value_counts().head(20)."
#     )
# # Aggregate daily earthquake counts for this region
# daily_counts_region = (
#     region_df.groupby("date").size().reset_index(name="count")
# )
# daily_counts_region["date"] = pd.to_datetime(daily_counts_region["date"])
# daily_counts_region = daily_counts_region.sort_values("date").reset_index(drop=True)

# if len(daily_counts_region) <= SEQ_LEN + 10:
#     raise ValueError(
#         f"Not enough days ({len(daily_counts_region)}) for SEQ_LEN={SEQ_LEN}. "
#         "Choose a region with more data or reduce SEQ_LEN."
#     )
# values_reg = daily_counts_region["count"].values.reshape(-1, 1)
# scaler_ts = MinMaxScaler()
# counts_scaled_reg = scaler_ts.fit_transform(values_reg)

# def create_sequences(data, seq_length):
#     Xs, ys = [], []
#     for i in range(len(data) - seq_length):
#         Xs.append(data[i : i + seq_length])
#         ys.append(data[i + seq_length])
#     return np.array(Xs), np.array(ys)

# X_ts, y_ts = create_sequences(counts_scaled_reg, SEQ_LEN)
# X_ts = X_ts.reshape((X_ts.shape[0], X_ts.shape[1], 1))  # (samples, timesteps, features)

# # Time-respecting split: first 80% train, last 20% test
# split_idx = int(len(X_ts) * 0.8)
# X_train_ts, X_test_ts = X_ts[:split_idx], X_ts[split_idx:]
# y_train_ts, y_test_ts = y_ts[:split_idx], y_ts[split_idx:]

# print(f"\nTrain sequences: {len(X_train_ts)}")
# print(f"Test sequences : {len(X_test_ts)}")

# model_lstm = Sequential(
#     [
#         LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
#         Dropout(0.2),
#         LSTM(32),
#         Dropout(0.2),
#         Dense(16, activation="relu"),
#         Dense(1),
#     ]
# )

# model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

# early_stop_lstm = EarlyStopping(
#     monitor="val_loss", patience=15, restore_best_weights=True
# )

# history_lstm = model_lstm.fit(
#     X_train_ts,
#     y_train_ts,
#     epochs=100,
#     batch_size=32,
#     validation_split=0.2,
#     callbacks=[early_stop_lstm],
#     verbose=1,
# )

# y_pred_scaled = model_lstm.predict(X_test_ts)

# # Invert scaling
# y_pred_counts = scaler_ts.inverse_transform(y_pred_scaled).flatten()
# y_test_counts = scaler_ts.inverse_transform(y_test_ts).flatten()
# y_train_counts = scaler_ts.inverse_transform(y_train_ts).flatten()

# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# lstm_mse = mean_squared_error(y_test_counts, y_pred_counts)
# lstm_rmse = np.sqrt(lstm_mse)
# lstm_mae = mean_absolute_error(y_test_counts, y_pred_counts)
# lstm_r2 = r2_score(y_test_counts, y_pred_counts)

# print(f"   RMSE: {lstm_rmse:.4f}")
# print(f"   MAE : {lstm_mae:.4f}")
# print(f"   R²  : {lstm_r2:.4f}")


# DIGITAL TWIN & REAL-TIME OPTIMIZATION
# I'll use the trained Random Forest classifier & regressor as the core DT models

rf_clf = classification_results["Random Forest"]["model"]
rf_reg = regression_results["Random Forest Regressor"]["model"]
risk_label_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}
def predict_risk_dt(features_array):
    """Predict risk level and probabilities for a single feature vector."""
    feats_scaled = scaler_clf.transform([features_array])
    pred_class = rf_clf.predict(feats_scaled)[0]
    proba = rf_clf.predict_proba(feats_scaled)[0]
    return pred_class, proba

def predict_magnitude_dt(features_array):
    """Predict magnitude for a single feature vector."""
    feats_scaled = scaler_reg.transform([features_array])
    pred_mag = rf_reg.predict(feats_scaled)[0]
    return pred_mag

def generate_recommendations(risk_label, magnitude, depth_km, location_str):
    """Generate simple optimization/response recommendations."""
    actions = []
    resources = {}
    alert_level = ""

    if risk_label == "HIGH" or magnitude >= 6.0:
        alert_level = "CRITICAL"
        actions = [
            "Activate emergency response protocols immediately",
            "Deploy search and rescue teams to affected region",
            "Issue public alerts and possible tsunami warning (if coastal)",
            "Pre-position medical supplies and shelters",
        ]
        resources = {
            "Emergency Personnel": "100%",
            "Medical Teams": "100%",
            "Rescue Equipment": "100%",
            "Shelter Capacity": "Maximum",
        }
    elif risk_label == "MEDIUM" or magnitude >= 4.5:
        alert_level = "WARNING"
        actions = [
            "Increase seismic monitoring frequency",
            "Place emergency services on standby",
            "Inspect critical infrastructure",
            "Issue public awareness advisory",
        ]
        resources = {
            "Emergency Personnel": "50%",
            "Medical Teams": "25%",
            "Rescue Equipment": "50%",
            "Shelter Capacity": "Standby",
        }
    else:
        alert_level = "NORMAL"
        actions = [
            "Continue routine monitoring",
            "Log event for historical analysis",
            "Update prediction models with new data",
        ]
        resources = {
            "Emergency Personnel": "10%",
            "Medical Teams": "5%",
            "Rescue Equipment": "10%",
            "Shelter Capacity": "Normal",
        }

    return {
        "alert_level": alert_level,
        "actions": actions,
        "resource_allocation": resources,
        "location": location_str,
        "estimated_magnitude": magnitude,
        "depth_km": depth_km,
    }


# Real-time data fetch
def fetch_realtime_earthquakes_hour():
    """Fetch latest earthquakes from USGS feed (last hour, M>=2.5)."""
    url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_hour.geojson"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            return resp.json()
    except Exception as e:
        print(f"Error fetching real-time data: {e}")
    return None


def build_feature_vector_from_event(event):
    """Build feature vector in the same order as feature_columns from a USGS event."""
    props = event["properties"]
    coords = event["geometry"]["coordinates"]

    eq_time = datetime.fromtimestamp(props["time"] / 1000)

    depth = coords[2] if len(coords) > 2 else 10.0
    lat = coords[1]
    lon = coords[0]

    feat_dict = {
        "depth": depth,
        "latitude": lat,
        "longitude": lon,
        "nst": props.get("nst", 10),
        "dmin": props.get("dmin", 0.5),
        "rms": props.get("rms", 0.5),
        "gap": props.get("gap", 100),
        "hour": eq_time.hour,
        "day_of_week": eq_time.weekday(),
        "month": eq_time.month,
        "dist_from_equator": abs(lat),
    }

    feature_array = [feat_dict[col] for col in feature_columns]
    return feature_array, eq_time

print("fetching real-time earthquakes (last hour, M≥2.5)")
rt_data = fetch_realtime_earthquakes_hour()

if not rt_data or len(rt_data.get("features", [])) == 0:
    print("No events in the last hour. Falling back to last 24 hours (M≥2.5)")
    backup_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson"
    resp = requests.get(backup_url)
    if resp.status_code == 200:
        rt_data = resp.json()

if rt_data and len(rt_data.get("features", [])) > 0:
    feats = rt_data["features"][:10]  # take up to 10 most recent
    print(f"Found {len(rt_data['features'])} events in feed; showing first {len(feats)}")

    for i, ev in enumerate(feats, start=1):
        props = ev["properties"]
        place = props.get("place", "Unknown location")
        actual_mag = props.get("mag", None)

        feat_array, eq_time = build_feature_vector_from_event(ev)

        # Guard against out-of-domain magnitudes (model trained on M≥2.5)
        if actual_mag is not None and actual_mag < 2.5:
            print(f"REAL-TIME EARTHQUAKE #{i}")
            print(f"Time: {eq_time}")
            print(f"Location: {place}")
            print(f"Actual Magnitude: {actual_mag}")
            print("Model trained only on events with M≥2.5; prediction skipped.")
            continue

        pred_class, proba = predict_risk_dt(feat_array)
        pred_label = risk_label_map.get(int(pred_class), "UNKNOWN")
        pred_mag = predict_magnitude_dt(feat_array)

        rec = generate_recommendations(
            risk_label=pred_label,
            magnitude=pred_mag,
            depth_km=feat_array[0],
            location_str=place,
        )

        print(f"REAL-TIME EARTHQUAKE #{i}")
        print(f"Time: {eq_time}")
        print(f"Location: {place}")
        print(f"Actual Magnitude (if reported): {actual_mag}")
        print(
            f"Predicted Magnitude: {pred_mag:.2f}"
        )
        print(
            f"Predicted Risk: {pred_label}  (P(High)={proba[2]*100:.1f}%, P(Med)={proba[1]*100:.1f}%)"
        )
        print(f"Depth: {feat_array[0]:.1f} km")
        print(f"Alert Level: {rec['alert_level']}")
        print("Recommended Actions (top 3):")
        for action in rec["actions"][:3]:
            print(f"      • {action}")
else:
    print("Unable to retrieve real-time events.")


   Retrieved 14640 events
   Retrieved 13326 events
   Retrieved 11447 events
   Retrieved 12989 events
       magnitude      depth  significance   latitude  longitude
count  52402.000  52402.000     52402.000  52402.000  52402.000
mean       3.896     61.290       247.432     22.009    -16.517
std        0.846    109.419       108.827     30.776    132.413
min        2.500     -3.500        96.000    -73.220   -179.999
25%        3.000     10.000       139.000     -2.764   -151.894
50%        4.200     17.614       271.000     28.951    -66.856
75%        4.500     60.088       312.000     51.262    128.054
max        8.800    671.043      2910.000     87.082    179.999

Stats by Risk Level (magnitude, depth, significance):
           magnitude                 depth                 significance  \
                mean    std  count    mean      std  count         mean   
risk_level                                                                
High           6.371  0.427    240  60.6

KeyboardInterrupt: 